In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import RadioButtons, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# LOW-PASS PROTOTYPE TO BAND-PASS FREQUENCY TRANSFORMATION
#
# Exercise specifications:
#
#       ωp1 = 9500 rad/s
#       ωp2 = 10500 rad/s
#       ωs1 = 5000 rad/s
#       ωs2 = 15000 rad/s
#       Ap  = 1.0 dB
#       As  = 50.0 dB
#
# Prototype:
#
#       Chebyshev II
#
# Frequency transformation:
#
#                       p² + ω0²
#       s = -----------------------------
#                         p B
#
# where:
#
#       ω0 = sqrt(ωp1 ωp2)
#
# Frequency mapping:
#
#                    ω² - ω0²
#       Ω = -----------------------------
#                         B ω
#
# Prototype-pole mapping:
#
#       p² - B sk p + ω0² = 0
#
# Thus every prototype pole produces TWO band-pass poles.
#
# Prototype finite-zero mapping:
#
#       p² - B zk p + ω0² = 0
#
# Thus every finite prototype zero produces TWO band-pass zeros.
#
# If the prototype contains N poles and M finite zeros, the transformation
# additionally produces N-M zeros at p = 0.
#
# Therefore:
#
#       prototype order = N
#       band-pass order  = 2N
#
# ==============================================================================
# DISPLAY STRATEGY
# ==============================================================================
#
# The prototype and transformed filters are displayed separately.
#
# Filter View:
#
#       Prototype LP
#       Transformed BP
#
# Displayed Quantity:
#
#       Pole-Zero Diagram
#       Magnitude Response
#       Phase Response
#       Group Delay
#       Impulse Response
#       Step Response
#
# The plot canvas remains 850 x 540 pixels.
#
# The plotting area is shifted slightly toward the LEFT inside the same canvas
# so that it lies closer to the numerical-information frame without overlapping
# it.
#
# The pole-zero legend is displayed BELOW the graph with its entries arranged
# horizontally on the same line.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}

</style>
"""))

# ==============================================================================
# EXERCISE SPECIFICATIONS
# ==============================================================================

wp1 = 9500.0
wp2 = 10500.0
ws1_original = 5000.0
ws2_original = 15000.0
Ap = 1.0
As = 50.0

# ==============================================================================
# STEP 1: INITIAL GEOMETRIC CENTER AND PASSBAND WIDTH
# ==============================================================================

omega0 = np.sqrt(wp1 * wp2)

B_initial = wp2 - wp1

# ==============================================================================
# STEP 2: STOPBAND REDEFINITION
# ==============================================================================

ws2_redefined = wp1 * wp2 / ws1_original

Omega_s_1 = (ws2_redefined - ws1_original) / B_initial

ws1_redefined = wp1 * wp2 / ws2_original

Omega_s_2 = (ws2_original - ws1_redefined) / B_initial

Omega_s_equivalent = min(Omega_s_1, Omega_s_2)

Omega_p_initial = 1.0

# ==============================================================================
# STEP 3: MINIMUM CHEBYSHEV-II ORDER
# ==============================================================================

ratio = np.sqrt((10.0**(As / 10.0) - 1.0) / (10.0**(Ap / 10.0) - 1.0))

N_exact = np.arccosh(ratio) / np.arccosh(Omega_s_equivalent / Omega_p_initial)

N = int(np.ceil(N_exact))

# ==============================================================================
# STEP 4: NORMALIZED CHEBYSHEV-II PROTOTYPE
# ==============================================================================

Omega_s = 1.0

Omega_p = 1.0 / np.cosh(np.arccosh(ratio) / N)

# ==============================================================================
# STEP 5: BAND-PASS BANDWIDTH PARAMETER
# ==============================================================================

B = (wp2 - wp1) / Omega_p

# ==============================================================================
# STEP 6: EFFECTIVE STOPBAND EDGES
# ==============================================================================

ws1_effective = -B * Omega_s / 2.0 + np.sqrt(omega0**2 + (B * Omega_s / 2.0)**2)

ws2_effective = +B * Omega_s / 2.0 + np.sqrt(omega0**2 + (B * Omega_s / 2.0)**2)

# ==============================================================================
# STEP 7: CHEBYSHEV-II AUXILIARY PARAMETERS
# ==============================================================================

epsilon = 1.0 / np.sqrt(10.0**(As / 10.0) - 1.0)

mu = np.arcsinh(1.0 / epsilon) / N

sinh_mu = np.sinh(mu)

cosh_mu = np.cosh(mu)

# ==============================================================================
# STEP 8: PROTOTYPE CHEBYSHEV-II POLES
# ==============================================================================

k = np.arange(1, N + 1)

theta = (2.0 * k - 1.0) * np.pi / (2.0 * N)

auxiliary_poles = -sinh_mu * np.sin(theta) + 1j * cosh_mu * np.cos(theta)

prototype_poles = 1.0 / auxiliary_poles

# ==============================================================================
# STEP 9: PROTOTYPE FINITE ZEROS
# ==============================================================================

cos_theta = np.cos(theta)

finite_mask = np.abs(cos_theta) > 1e-10

prototype_zeros = 1j / cos_theta[finite_mask]

num_prototype_zeros_at_infinity = N - len(prototype_zeros)

# ==============================================================================
# STEP 10: PROTOTYPE GAIN
# ==============================================================================

prototype_gain = np.real_if_close(np.prod(-prototype_poles) / np.prod(-prototype_zeros), tol=1000).real

# ==============================================================================
# STEP 11: PROTOTYPE TRANSFER FUNCTION
# ==============================================================================

prototype_num, prototype_den = signal.zpk2tf(prototype_zeros, prototype_poles, prototype_gain)

prototype_num = np.real_if_close(prototype_num, tol=1000).real

prototype_den = np.real_if_close(prototype_den, tol=1000).real

# ==============================================================================
# STEP 12: BAND-PASS POLE MAPPING
# ==============================================================================

transformed_poles_list = []

for prototype_pole in prototype_poles:

    discriminant = complex((B * prototype_pole)**2 - 4.0 * omega0**2)

    root_discriminant = np.sqrt(discriminant)

    transformed_poles_list.append((B * prototype_pole + root_discriminant) / 2.0)

    transformed_poles_list.append((B * prototype_pole - root_discriminant) / 2.0)

transformed_poles = np.asarray(transformed_poles_list, dtype=complex)

# ==============================================================================
# STEP 13: BAND-PASS ZERO MAPPING
# ==============================================================================

transformed_zeros_list = []

for prototype_zero in prototype_zeros:

    discriminant = complex((B * prototype_zero)**2 - 4.0 * omega0**2)

    root_discriminant = np.sqrt(discriminant)

    transformed_zeros_list.append((B * prototype_zero + root_discriminant) / 2.0)

    transformed_zeros_list.append((B * prototype_zero - root_discriminant) / 2.0)

for index in range(N - len(prototype_zeros)):

    transformed_zeros_list.append(0.0 + 0.0j)

transformed_zeros = np.asarray(transformed_zeros_list, dtype=complex)

# ==============================================================================
# STEP 14: BAND-PASS GAIN
# ==============================================================================

M = len(prototype_zeros)

transformed_gain = prototype_gain * B**(N - M)

# ==============================================================================
# STEP 15: BAND-PASS TRANSFER FUNCTION
# ==============================================================================

transformed_num, transformed_den = signal.zpk2tf(transformed_zeros, transformed_poles, transformed_gain)

transformed_num = np.real_if_close(transformed_num, tol=1000).real

transformed_den = np.real_if_close(transformed_den, tol=1000).real

# ==============================================================================
# FREQUENCY AXES
# ==============================================================================

Omega_axis = np.logspace(-3, 2, 6000)

omega_axis = np.logspace(3, 4.5, 7000)

# ==============================================================================
# PROTOTYPE FREQUENCY RESPONSE
# ==============================================================================

_, H_prototype = signal.freqs(prototype_num, prototype_den, worN=Omega_axis)

prototype_magnitude = np.abs(H_prototype)

prototype_phase = np.unwrap(np.angle(H_prototype))

prototype_phase_deg = np.rad2deg(prototype_phase)

prototype_group_delay = -np.gradient(prototype_phase, Omega_axis)

# ==============================================================================
# TRANSFORMED BAND-PASS FREQUENCY RESPONSE
# ==============================================================================

_, H_transformed = signal.freqs(transformed_num, transformed_den, worN=omega_axis)

transformed_magnitude = np.abs(H_transformed)

transformed_phase = np.unwrap(np.angle(H_transformed))

transformed_phase_deg = np.rad2deg(transformed_phase)

transformed_group_delay = -np.gradient(transformed_phase, omega_axis)

# ==============================================================================
# TIME-DOMAIN RESPONSES
# ==============================================================================

prototype_system = signal.TransferFunction(prototype_num, prototype_den)

transformed_system = signal.TransferFunction(transformed_num, transformed_den)

prototype_slowest_rate = np.min(np.abs(np.real(prototype_poles)))

prototype_time_constant = 1.0 / prototype_slowest_rate

t_prototype_max = 12.0 * prototype_time_constant

t_prototype = np.linspace(0.0, t_prototype_max, 6000)

transformed_slowest_rate = np.min(np.abs(np.real(transformed_poles)))

transformed_time_constant = 1.0 / transformed_slowest_rate

t_transformed_max = 12.0 * transformed_time_constant

t_transformed = np.linspace(0.0, t_transformed_max, 7000)

t_impulse_prototype, h_prototype = signal.impulse(prototype_system, T=t_prototype)

t_step_prototype, step_prototype = signal.step(prototype_system, T=t_prototype)

t_impulse_transformed, h_transformed = signal.impulse(transformed_system, T=t_transformed)

t_step_transformed, step_transformed = signal.step(transformed_system, T=t_transformed)

# ==============================================================================
# ATTENUATION CALCULATION
# ==============================================================================

def attenuation_from_transfer_function(b, a, frequency):

    _, H = signal.freqs(b, a, worN=np.asarray([frequency]))

    magnitude = max(np.abs(H[0]), 1e-15)

    return -20.0 * np.log10(magnitude)

# ==============================================================================
# SPECIFICATION VERIFICATION
# ==============================================================================

Ap1_actual = attenuation_from_transfer_function(transformed_num, transformed_den, wp1)

Ap2_actual = attenuation_from_transfer_function(transformed_num, transformed_den, wp2)

As1_actual = attenuation_from_transfer_function(transformed_num, transformed_den, ws1_original)

As2_actual = attenuation_from_transfer_function(transformed_num, transformed_den, ws2_original)

# ==============================================================================
# POLYNOMIAL STRING
# ==============================================================================

def polynomial_string(coefficients, variable='p'):

    degree = len(coefficients) - 1

    terms = []

    for index, coefficient in enumerate(coefficients):

        power = degree - index

        if abs(coefficient) < 1e-8:

            continue

        if power == 0:

            terms.append(f'{coefficient:.6e}')

        elif power == 1:

            terms.append(f'{coefficient:.6e}{variable}')

        else:

            terms.append(f'{coefficient:.6e}{variable}^{power}')

    return ' + '.join(terms)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1580px;
    max-width:1580px;
    box-sizing:border-box;
">

<b>Low-Pass Prototype to Band-Pass Frequency Transformation</b><br>

A Chebyshev-II prototype is constructed for a band-pass filter with
ω<sub>p1</sub> = {wp1:.0f} rad/s,
ω<sub>p2</sub> = {wp2:.0f} rad/s,
ω<sub>s1</sub> = {ws1_original:.0f} rad/s,
ω<sub>s2</sub> = {ws2_original:.0f} rad/s,
A<sub>p</sub> = {Ap:.1f} dB and
A<sub>s</sub> = {As:.0f} dB.

The transformation is

<b>s = (p² + ω<sub>0</sub>²)/(pB)</b>.

<br>

<b>Visualization strategy:</b>
The normalized prototype and the transformed physical band-pass filter are
displayed separately because their frequency, pole and time scales differ
substantially. Use the <b>Filter View</b> selector to inspect either filter
with axis limits appropriate to its own scale.

</div>
""", layout=Layout(width='1590px', max_width='1590px'))

# ==============================================================================
# FILTER-VIEW SELECTOR
# ==============================================================================

filter_view_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Filter View
</div>
""")

filter_view_selector = RadioButtons(options=['Prototype LP', 'Transformed BP'], value='Prototype LP', description='', layout=Layout(width='175px'))

filter_view_panel = VBox([filter_view_title, filter_view_selector], layout=Layout(width='210px', min_width='210px', max_width='210px', border='1px solid #cccccc', padding='9px', align_items='flex-start'))

# ==============================================================================
# DISPLAY SELECTOR
# ==============================================================================

display_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Displayed Quantity
</div>
""")

display_selector = RadioButtons(options=['Pole-Zero Diagram', 'Magnitude Response', 'Phase Response', 'Group Delay', 'Impulse Response', 'Step Response'], value='Pole-Zero Diagram', description='', layout=Layout(width='185px'))

display_panel = VBox([display_title, display_selector], layout=Layout(width='210px', min_width='210px', max_width='210px', border='1px solid #cccccc', padding='9px', align_items='flex-start'))

# ==============================================================================
# INFORMATION PANELS
# ==============================================================================

info_left = HTML(layout=Layout(width='310px', max_width='310px'))

info_right = HTML(layout=Layout(width='330px', max_width='330px'))

# ==============================================================================
# INFORMATION-PANEL CONTENT
# ==============================================================================

prototype_pole_text = '<br>'.join([f's{k + 1} = {pole.real:+.6f} {pole.imag:+.6f}j' for k, pole in enumerate(prototype_poles)])

prototype_zero_text = '<br>'.join([f'z{k + 1} = {zero.real:+.6f} {zero.imag:+.6f}j' for k, zero in enumerate(prototype_zeros)])

transformed_pole_text = '<br>'.join([f'p{k + 1} = {pole.real:+.3f} {pole.imag:+.3f}j' for k, pole in enumerate(transformed_poles)])

transformed_zero_text = '<br>'.join([f'z{k + 1} = {zero.real:+.3f} {zero.imag:+.3f}j' for k, zero in enumerate(transformed_zeros)])

numerator_text = polynomial_string(transformed_num)

denominator_text = polynomial_string(transformed_den)

Ap1_status = '✓ satisfied' if Ap1_actual <= Ap + 1e-8 else '✗ not satisfied'

Ap2_status = '✓ satisfied' if Ap2_actual <= Ap + 1e-8 else '✗ not satisfied'

As1_status = '✓ satisfied' if As1_actual >= As - 1e-8 else '✗ not satisfied'

As2_status = '✓ satisfied' if As2_actual >= As - 1e-8 else '✗ not satisfied'

# ==============================================================================
# LEFT NUMERICAL COLUMN
# ==============================================================================

info_left.value = f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:9px 10px;
    font-size:11.1px;
    line-height:1.48;
    background:white;
    width:305px;
    box-sizing:border-box;
">

<b>Step 1 — Specifications</b><br>

<span style="color:#0066cc;">
ωp1 = {wp1:.0f}, ωp2 = {wp2:.0f} rad/s<br>
ωs1 = {ws1_original:.0f}, ωs2 = {ws2_original:.0f} rad/s<br>
Ap = {Ap:.1f} dB, As = {As:.0f} dB
</span>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 2 — Initial frequency mapping</b><br>

ω₀ = √(ωp1ωp2) =
<span style="color:#0066cc;">{omega0:.6f} rad/s</span><br>

B₀ = ωp2−ωp1 =
<span style="color:#0066cc;">{B_initial:.6f} rad/s</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 3 — Stopband redefinition</b><br>

ωs2′ =
<span style="color:#0066cc;">{ws2_redefined:.6f}</span><br>

Ωs⁽¹⁾ =
<span style="color:#0066cc;">{Omega_s_1:.6f}</span><br>

ωs1′ =
<span style="color:#0066cc;">{ws1_redefined:.6f}</span><br>

Ωs⁽²⁾ =
<span style="color:#0066cc;">{Omega_s_2:.6f}</span><br>

Critical Ωs =
<span style="color:#0066cc;"><b>{Omega_s_equivalent:.6f}</b></span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 4 — Minimum Chebyshev-II order</b><br>

Nmin =
<span style="color:#0066cc;">{N_exact:.6f}</span><br>

N =
<span style="color:#0066cc;"><b>{N}</b></span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 5 — Normalized prototype</b><br>

Ωs =
<span style="color:#0066cc;">{Omega_s:.6f}</span><br>

Ωp =
<span style="color:#0066cc;">{Omega_p:.6f}</span><br>

ε =
<span style="color:#0066cc;">{epsilon:.6e}</span><br>

μ =
<span style="color:#0066cc;">{mu:.6f}</span><br>

sinh(μ) =
<span style="color:#0066cc;">{sinh_mu:.6f}</span><br>

cosh(μ) =
<span style="color:#0066cc;">{cosh_mu:.6f}</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 6 — Band-pass transformation parameters</b><br>

ω₀ =
<span style="color:#0066cc;">{omega0:.6f} rad/s</span><br>

B =
<span style="color:#0066cc;"><b>{B:.6f} rad/s</b></span><br>

Effective ωs1 =
<span style="color:#0066cc;">{ws1_effective:.6f}</span><br>

Effective ωs2 =
<span style="color:#0066cc;">{ws2_effective:.6f}</span>

</div>

</div>
"""

# ==============================================================================
# RIGHT NUMERICAL COLUMN
# ==============================================================================

info_right.value = f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:9px 10px;
    font-size:11.1px;
    line-height:1.48;
    background:white;
    width:325px;
    box-sizing:border-box;
">

<b>Step 7 — Prototype poles</b><br>

<span style="color:#0066cc;">
{prototype_pole_text}
</span>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 8 — Prototype finite zeros</b><br>

<span style="color:#0066cc;">
{prototype_zero_text}
</span><br>

Zeros at infinity:
<span style="color:#0066cc;">{num_prototype_zeros_at_infinity}</span><br>

Prototype gain:
<span style="color:#0066cc;">{prototype_gain:.9f}</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 9 — Transformed poles</b><br>

<span style="color:#0066cc;">
{transformed_pole_text}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 10 — Transformed zeros</b><br>

<span style="color:#0066cc;">
{transformed_zero_text}
</span><br>

Band-pass order:
<span style="color:#0066cc;"><b>{2 * N}</b></span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 11 — Final transfer function</b><br>

H<sub>BP</sub>(p) = N(p) / D(p)<br><br>

<span style="color:#0066cc;">
N(p) = {numerator_text}<br><br>
D(p) = {denominator_text}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 12 — Specification verification</b><br>

A(ωp1) =
<span style="color:#0066cc;">{Ap1_actual:.6f} dB</span>
→ {Ap1_status}<br>

A(ωp2) =
<span style="color:#0066cc;">{Ap2_actual:.6f} dB</span>
→ {Ap2_status}<br>

A(ωs1) =
<span style="color:#0066cc;">{As1_actual:.6f} dB</span>
→ {As1_status}<br>

A(ωs2) =
<span style="color:#0066cc;">{As2_actual:.6f} dB</span>
→ {As2_status}

</div>

</div>
"""

# ==============================================================================
# MAIN FIGURE
#
# Canvas dimensions remain EXACTLY:
#
#       850 x 540 pixels
#
# The plotting rectangle itself retains approximately the original width,
# but is shifted toward the LEFT inside the canvas.
#
# Original plotting width:
#
#       0.76 - 0.12 = 0.64
#
# New plotting width:
#
#       0.715 - 0.075 = 0.64
#
# Therefore the graph is simply translated toward the left; it is not enlarged.
#
# More room is reserved underneath for the horizontal legend.
# ==============================================================================

fig, ax = plt.subplots(figsize=(8.5, 5.5))

response_line, = ax.plot([], [], linewidth=2.3, color='red')

pole_line, = ax.plot([], [], 'ro', markersize=7)

zero_line, = ax.plot([], [], 'gx', markersize=9, markeredgewidth=1.8)

horizontal_axis = ax.axhline(0.0, color='black', linewidth=0.8)

vertical_axis = ax.axvline(0.0, color='black', linewidth=0.8)

passband_line_1 = ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)

passband_line_2 = ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)

stopband_line_1 = ax.axvline(1.0, color='gray', linestyle=':', linewidth=1.0)

stopband_line_2 = ax.axvline(1.0, color='gray', linestyle=':', linewidth=1.0)

ax.grid(True, linestyle=':', alpha=0.5)

ax.tick_params(axis='both', labelsize=9)

# Shift the plot LEFT inside the same canvas while preserving its width.
# Extra bottom room is reserved for the legend.

fig.subplots_adjust(left=0.075, right=0.715, bottom=0.27, top=0.88)

fig.canvas.header_visible = False

fig.canvas.toolbar_visible = False

fig.canvas.resizable = False

fig.canvas.layout.width = '850px'

fig.canvas.layout.height = '540px'

# ==============================================================================
# LEGEND FUNCTION
#
# The legend is below the graph.
# The two entries are arranged horizontally.
# ==============================================================================

def external_legend(handles, labels):

    ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.18), borderaxespad=0.0, fontsize=8, frameon=True, title='Legend', title_fontsize=9, labelspacing=1.0, handlelength=2.5, ncol=len(labels), columnspacing=2.2)

# ==============================================================================
# MAIN UPDATE FUNCTION
# ==============================================================================

def update_plot(change=None):

    view = filter_view_selector.value

    selected = display_selector.value

    # --------------------------------------------------------------------------
    # RESET VISIBILITY
    # --------------------------------------------------------------------------

    response_line.set_visible(False)

    pole_line.set_visible(False)

    zero_line.set_visible(False)

    horizontal_axis.set_visible(False)

    vertical_axis.set_visible(False)

    passband_line_1.set_visible(False)

    passband_line_2.set_visible(False)

    stopband_line_1.set_visible(False)

    stopband_line_2.set_visible(False)

    old_legend = ax.get_legend()

    if old_legend is not None:

        old_legend.remove()

    # ==========================================================================
    # PROTOTYPE LOW-PASS FILTER
    # ==========================================================================

    if view == 'Prototype LP':

        response_line.set_color('blue')

        pole_line.set_color('blue')

        # ----------------------------------------------------------------------
        # POLE-ZERO DIAGRAM
        # ----------------------------------------------------------------------

        if selected == 'Pole-Zero Diagram':

            pole_line.set_visible(True)

            zero_line.set_visible(True)

            horizontal_axis.set_visible(True)

            vertical_axis.set_visible(True)

            pole_line.set_data(np.real(prototype_poles), np.imag(prototype_poles))

            zero_line.set_data(np.real(prototype_zeros), np.imag(prototype_zeros))

            all_values = np.concatenate([prototype_poles, prototype_zeros])

            real_limit = max(np.max(np.abs(np.real(all_values))), 1.0)

            imag_limit = max(np.max(np.abs(np.imag(all_values))), 1.0)

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(-1.25 * real_limit, 0.25 * real_limit)

            ax.set_ylim(-1.20 * imag_limit, 1.20 * imag_limit)

            ax.set_aspect('auto')

            ax.set_xlabel('Re{s}', fontsize=10)

            ax.set_ylabel('Im{s}', fontsize=10)

            ax.set_title('Prototype Chebyshev-II Pole-Zero Diagram', fontsize=13, fontweight='bold', pad=8)

            external_legend([pole_line, zero_line], ['Prototype poles', 'Finite prototype zeros'])

        # ----------------------------------------------------------------------
        # MAGNITUDE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Magnitude Response':

            response_line.set_visible(True)

            passband_line_1.set_visible(True)

            stopband_line_1.set_visible(True)

            response_line.set_data(Omega_axis, prototype_magnitude)

            passband_line_1.set_xdata([Omega_p, Omega_p])

            stopband_line_1.set_xdata([Omega_s, Omega_s])

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(1e-3, 100.0)

            ax.set_ylim(0.0, 1.08)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('|H_LPP(jΩ)|', fontsize=10)

            ax.set_title('Prototype Chebyshev-II Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # PHASE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Phase Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(Omega_axis, prototype_phase_deg)

            phase_min = np.min(prototype_phase_deg)

            phase_max = np.max(prototype_phase_deg)

            phase_margin = 0.06 * max(phase_max - phase_min, 90.0)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(1e-3, 100.0)

            ax.set_ylim(phase_min - phase_margin, phase_max + phase_margin)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('Phase (degrees)', fontsize=10)

            ax.set_title('Prototype Chebyshev-II Phase Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # GROUP DELAY
        # ----------------------------------------------------------------------

        elif selected == 'Group Delay':

            response_line.set_visible(True)

            response_line.set_data(Omega_axis, prototype_group_delay)

            finite_gd = prototype_group_delay[np.isfinite(prototype_group_delay)]

            finite_gd = finite_gd[finite_gd >= 0.0]

            gd_max = max(np.max(finite_gd), 1e-6)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(1e-3, 100.0)

            ax.set_ylim(0.0, 1.10 * gd_max)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('Group Delay', fontsize=10)

            ax.set_title('Prototype Chebyshev-II Group Delay', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # IMPULSE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Impulse Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_impulse_prototype, h_prototype)

            y_min = np.min(h_prototype)

            y_max = np.max(h_prototype)

            y_margin = 0.10 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_prototype_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t', fontsize=10)

            ax.set_ylabel('h_LPP(t)', fontsize=10)

            ax.set_title('Prototype Chebyshev-II Impulse Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # STEP RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Step Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_step_prototype, step_prototype)

            y_min = min(0.0, np.min(step_prototype))

            y_max = max(1.0, np.max(step_prototype))

            y_margin = 0.08 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_prototype_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t', fontsize=10)

            ax.set_ylabel('Step Response', fontsize=10)

            ax.set_title('Prototype Chebyshev-II Step Response', fontsize=13, fontweight='bold', pad=8)

    # ==========================================================================
    # TRANSFORMED BAND-PASS FILTER
    # ==========================================================================

    elif view == 'Transformed BP':

        response_line.set_color('red')

        pole_line.set_color('red')

        # ----------------------------------------------------------------------
        # POLE-ZERO DIAGRAM
        # ----------------------------------------------------------------------

        if selected == 'Pole-Zero Diagram':

            pole_line.set_visible(True)

            zero_line.set_visible(True)

            horizontal_axis.set_visible(True)

            vertical_axis.set_visible(True)

            pole_line.set_data(np.real(transformed_poles), np.imag(transformed_poles))

            zero_line.set_data(np.real(transformed_zeros), np.imag(transformed_zeros))

            all_values = np.concatenate([transformed_poles, transformed_zeros])

            real_limit = max(np.max(np.abs(np.real(all_values))), 1.0)

            imag_limit = max(np.max(np.abs(np.imag(all_values))), 1.0)

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(-1.35 * real_limit, 0.35 * real_limit)

            ax.set_ylim(-1.10 * imag_limit, 1.10 * imag_limit)

            ax.set_aspect('auto')

            ax.set_xlabel('Re{p}', fontsize=10)

            ax.set_ylabel('Im{p}', fontsize=10)

            ax.set_title('Transformed Band-Pass Pole-Zero Diagram', fontsize=13, fontweight='bold', pad=8)

            external_legend([pole_line, zero_line], ['Band-pass poles', 'Band-pass zeros'])

        # ----------------------------------------------------------------------
        # MAGNITUDE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Magnitude Response':

            response_line.set_visible(True)

            passband_line_1.set_visible(True)

            passband_line_2.set_visible(True)

            stopband_line_1.set_visible(True)

            stopband_line_2.set_visible(True)

            response_line.set_data(omega_axis, transformed_magnitude)

            passband_line_1.set_xdata([wp1, wp1])

            passband_line_2.set_xdata([wp2, wp2])

            stopband_line_1.set_xdata([ws1_original, ws1_original])

            stopband_line_2.set_xdata([ws2_original, ws2_original])

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(3000.0, 25000.0)

            ax.set_ylim(0.0, 1.08)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('|H_BP(jω)|', fontsize=10)

            ax.set_title('Transformed Band-Pass Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # PHASE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Phase Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(omega_axis, transformed_phase_deg)

            visible_mask = (omega_axis >= 3000.0) & (omega_axis <= 25000.0)

            visible_phase = transformed_phase_deg[visible_mask]

            phase_min = np.min(visible_phase)

            phase_max = np.max(visible_phase)

            phase_margin = 0.06 * max(phase_max - phase_min, 90.0)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(3000.0, 25000.0)

            ax.set_ylim(phase_min - phase_margin, phase_max + phase_margin)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('Phase (degrees)', fontsize=10)

            ax.set_title('Transformed Band-Pass Phase Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # GROUP DELAY
        # ----------------------------------------------------------------------

        elif selected == 'Group Delay':

            response_line.set_visible(True)

            response_line.set_data(omega_axis, transformed_group_delay)

            visible_mask = (omega_axis >= 3000.0) & (omega_axis <= 25000.0)

            finite_gd = transformed_group_delay[visible_mask]

            finite_gd = finite_gd[np.isfinite(finite_gd)]

            finite_gd = finite_gd[finite_gd >= 0.0]

            gd_max = max(np.max(finite_gd), 1e-8)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(3000.0, 25000.0)

            ax.set_ylim(0.0, 1.10 * gd_max)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('Group Delay (s)', fontsize=10)

            ax.set_title('Transformed Band-Pass Group Delay', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # IMPULSE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Impulse Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_impulse_transformed, h_transformed)

            y_min = np.min(h_transformed)

            y_max = np.max(h_transformed)

            y_margin = 0.08 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_transformed_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t (s)', fontsize=10)

            ax.set_ylabel('h_BP(t)', fontsize=10)

            ax.set_title('Transformed Band-Pass Impulse Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # STEP RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Step Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_step_transformed, step_transformed)

            y_min = np.min(step_transformed)

            y_max = np.max(step_transformed)

            y_range = max(y_max - y_min, 1e-8)

            y_margin = 0.08 * y_range

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_transformed_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t (s)', fontsize=10)

            ax.set_ylabel('Step Response', fontsize=10)

            ax.set_title('Transformed Band-Pass Step Response', fontsize=13, fontweight='bold', pad=8)

    # --------------------------------------------------------------------------
    # ZERO-REFERENCE AXIS
    # --------------------------------------------------------------------------

    horizontal_axis.set_ydata([0.0, 0.0])

    # --------------------------------------------------------------------------
    # GRID
    # --------------------------------------------------------------------------

    ax.grid(True, which='both', linestyle=':', alpha=0.5)

    # --------------------------------------------------------------------------
    # REDRAW
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

filter_view_selector.observe(update_plot, names='value')

display_selector.observe(update_plot, names='value')

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_plot()

# ==============================================================================
# LEFT CONTROL COLUMN
# ==============================================================================

left_column = VBox([filter_view_panel, display_panel], layout=Layout(width='220px', min_width='220px', max_width='220px', align_items='flex-start'))

# ==============================================================================
# NUMERICAL COLUMNS
# ==============================================================================

information_left_column = VBox([info_left], layout=Layout(width='315px', min_width='315px', max_width='315px', align_items='flex-start'))

information_right_column = VBox([info_right], layout=Layout(width='335px', min_width='335px', max_width='335px', align_items='flex-start'))

# ==============================================================================
# PLOT COLUMN
#
# IMPORTANT:
#
# NO negative margin.
#
# Therefore the canvas cannot enter the numerical-information frame.
# ==============================================================================

plot_column = VBox([fig.canvas], layout=Layout(width='850px', min_width='850px', max_width='850px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# MAIN LAYOUT
# ==============================================================================

main_layout = HBox([left_column, information_left_column, information_right_column, plot_column], layout=Layout(width='1730px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)